# Research 27 — Combine region interactions

**Hypothesis:** loyalty × region and engagement × region may add complementary region-conditioned signals. Test one fixed 80/10/10 mixture of current reference and the two OOF candidates, against the respective 80/20 single-interaction blends. No fitting or weight search; Test is not loaded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    values = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(values).all() and (values >= 0).all(), filename
    return values / values.sum(axis=1, keepdims=True)


p_base = load_probs("segment-oof.csv")
p_loyalty_region = load_probs("segment-loyalty-region-balanced-oof.csv")
p_engagement_region = load_probs("segment-engagement-region-balanced-oof.csv")
candidates = {
    "Current reference": p_base,
    "80/20 loyalty × region": 0.8 * p_base + 0.2 * p_loyalty_region,
    "80/20 engagement × region": 0.8 * p_base + 0.2 * p_engagement_region,
    "80/10/10 both region interactions": 0.8 * p_base
    + 0.1 * p_loyalty_region
    + 0.1 * p_engagement_region,
}
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
base_folds = results.loc[
    results.candidate == "Current reference", [f"fold_{fold}_macro_f1" for fold in range(1, 4)]
].iloc[0]
for fold in range(1, 4):
    results[f"fold_{fold}_delta"] = (
        results[f"fold_{fold}_macro_f1"] - base_folds[f"fold_{fold}_macro_f1"]
    )
display(results.round(6))
results.to_csv(ROOT / "assets/segment-region-interactions-threeway.csv", index=False)
pd.DataFrame(
    candidates["80/10/10 both region interactions"], columns=[f"prob_{i}" for i in range(4)]
).assign(segment=y).to_csv(
    ROOT / "predictions/segment-region-interactions-threeway-oof.csv", index=False
)
print("Fixed region-interaction synthesis complete; Test not loaded.")

,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1,fold_1_delta,fold_2_delta,fold_3_delta
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.000000,0.000000,0.000000
1,80/20 loyalty × region,0.751473,0.764911,0.750107,0.750578,0.753706,0.000661,0.000757,0.000316
2,80/20 engagement × region,0.751401,0.764838,0.749597,0.751161,0.753416,0.000151,0.001340,0.000026
3,80/10/10 both region interactions,0.751332,0.764822,0.749413,0.750709,0.753843,-0.000032,0.000888,0.000454


Fixed region-interaction synthesis complete; Test not loaded.
